# NCAA Bracket Model — Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def show_columns(df):
    for col in df.columns:
        print(col)

## Step 1: Build Team Season Averages

In [ ]:
season_stats = pd.read_csv("../data/raw/MRegularSeasonDetailedResults.csv")
season_stats.head()

In [ ]:
show_columns(season_stats)

In [ ]:
winners = season_stats[['Season', 'WTeamID', 'WScore', 'LScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']]
winners.rename(columns={'WTeamID': 'TeamID', 'WScore': 'Score', 'LScore': 'OppScore', 'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3', 'WFTM': 'FTM', 'WFTA': 'FTA', 'WAst': 'Ast', 'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF'}, inplace=True)
winners['Win'] = 1
winners.head()

In [ ]:
losers = season_stats[['Season', 'LTeamID', 'LScore', 'WScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']]
losers.rename(columns={'LTeamID': 'TeamID', 'LScore': 'Score', 'WScore': 'OppScore', 'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3', 'LFTM': 'FTM', 'LFTA': 'FTA', 'LAst': 'Ast', 'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF'}, inplace=True)
losers['Win'] = 0
losers.head()

In [ ]:
team_stats = pd.concat([winners, losers], ignore_index=True)
team_stats.head()

In [ ]:
# Verifying the proper amount of rows
print(f"Number of rows in winners: {len(winners)}\nNumber of rows in losers: {len(losers)}\nTotal rows in team_stats: {len(team_stats)}")
if len(winners) + len(losers) == len(team_stats) and len(season_stats) * 2 == len(team_stats):
    print("The number of rows in team_stats is correct.")


In [ ]:
season_averages = team_stats.groupby(['Season', 'TeamID']).mean().reset_index().sort_values(by=['Season', 'TeamID'])    
season_averages.head()

In [ ]:
season_averages.shape

In [ ]:
checkwin = season_averages[(season_averages['Win'] < 0) | (season_averages['Win'] > 1)]
if len(checkwin) > 0:
    print("There are invalid values in the 'Win' column.")
else:
    print("All values in the 'Win' column are valid (0 or 1).")

In [ ]:
print(season_averages.shape)
season_averages

## Step 2: Clean and Build Seed Data

In [ ]:
seeds = pd.read_csv("../data/raw/MNCAATourneySeeds.csv")
seeds.head()

In [ ]:
seed_clean = seeds.copy()
seed_clean['Seed'] = seed_clean['Seed'].str.replace(r'[a-zA-Z]', '', regex=True).astype(int)
seed_clean = seed_clean[['Season', 'TeamID', 'Seed']]
seed_clean.head()

In [ ]:
if (seed_clean['Seed'] <= 16).all() and (seed_clean['Seed'] >= 1).all():
    print("All seed values are valid (between 1 and 16).")

In [ ]:
print(seed_clean.shape)
seed_clean

## Step 3: Build Rankings Table (Massey)

In [ ]:
massey = pd.read_csv("../data/raw/MMasseyOrdinals.csv")
massey.head()

In [ ]:
massey_clean = massey.copy()
massey_clean = massey_clean[(massey_clean['RankingDayNum'] == 133) & massey_clean['SystemName'].isin(['POM', 'SAG', 'MOR', 'DUN'])].reset_index(drop=True)
massey_clean = massey_clean[['Season', 'TeamID', 'SystemName', 'OrdinalRank']]      
massey_clean.head()


In [ ]:
massey_pivot = massey_clean.pivot(index = ['Season', 'TeamID'], columns = 'SystemName', values = 'OrdinalRank').reset_index()
massey_pivot.columns.name = None
massey_pivot.head()

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
# Cleaning up massey data
massey_pivot['DUN'] = massey_pivot.groupby('Season')['DUN'].transform(lambda x: x.fillna(x.median())) 
massey_pivot['MOR'] = massey_pivot.groupby('Season')['MOR'].transform(lambda x: x.fillna(x.median()))
massey_pivot['POM'] = massey_pivot.groupby('Season')['POM'].transform(lambda x: x.fillna(x.median()))
massey_pivot['SAG'] = massey_pivot.groupby('Season')['SAG'].transform(lambda x: x.fillna(x.median()))
massey_pivot.head() 

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
massey_pivot.groupby('Season')['DUN'].count()

In [ ]:
massey_pivot.groupby('Season')['SAG'].count()

In [ ]:
#Handeling Missing Values in DUN
dun_median = massey_pivot.groupby('Season')['DUN'].median().sort_index().ffill()
print(dun_median)
season_medians = massey_pivot['Season'].map(dun_median)
massey_pivot['DUN'] = massey_pivot['DUN'].fillna(season_medians)
massey_pivot.isnull().sum()

In [ ]:
#Handeling Missing Values in SAG
sag_median = massey_pivot.groupby('Season')['SAG'].median().sort_index().ffill()
print(sag_median)
season_medians = massey_pivot['Season'].map(sag_median)
massey_pivot['SAG'] = massey_pivot['SAG'].fillna(season_medians)
massey_pivot.isnull().sum()

In [ ]:
print(massey_pivot.shape)
massey_pivot

## Step 4: Build Tournament History Features

##### Seed Features

In [ ]:
tourney_results = pd.read_csv("../data/raw/MNCAATourneyCompactResults.csv")
tourney_results.head()

In [ ]:
bracket_winners = tourney_results[['Season', 'WTeamID', 'DayNum']]
bracket_winners.rename(columns={'WTeamID': 'TeamID'}, inplace=True)

bracket_losers = tourney_results[['Season', 'LTeamID', 'DayNum']]
bracket_losers.rename(columns={'LTeamID': 'TeamID'}, inplace=True)

In [ ]:
tourney_appearances = pd.concat([bracket_winners, bracket_losers], ignore_index=True)
tourney_appearances.head()

In [ ]:
deepest_game = tourney_appearances.groupby(['Season', 'TeamID'])['DayNum'].max().reset_index()
deepest_game

In [ ]:
conditions = [
    deepest_game['DayNum'] == 154,
    deepest_game['DayNum'] == 152,
    (deepest_game['DayNum'] >= 145) & (deepest_game['DayNum'] < 152),
    (deepest_game['DayNum'] >= 143) & (deepest_game['DayNum'] < 145),
    (deepest_game['DayNum'] >= 138) & (deepest_game['DayNum'] < 143),
    (deepest_game['DayNum'] >= 134) & (deepest_game['DayNum'] < 138),
    deepest_game['DayNum'] < 134
]

values = [6, 5, 4, 3, 2, 1, 0]

deepest_game['Round'] = np.select(conditions, values, default=0)
deepest_game

In [ ]:
# Verifying the distribution of rounds
print(deepest_game['Round'].value_counts())
print(deepest_game['Round'].max())

In [ ]:
all_teams = deepest_game['TeamID'].unique()
all_seasons = deepest_game['Season'].unique()
possible_indices = pd.MultiIndex.from_product([all_seasons, all_teams], names=['Season', 'TeamID'])
possible_indices = possible_indices.to_frame(index=False)

In [ ]:
tournament_history = pd.merge(possible_indices, deepest_game, on=['Season', 'TeamID'], how='left')
tournament_history = tournament_history.fillna({'Round': 0})
tournament_history

In [ ]:
tournament_history['round_last_year'] = tournament_history.groupby('TeamID')['Round'].shift(1)
tournament_history['round_avg_3yr'] = tournament_history.groupby('TeamID')['Round'].transform(lambda x: x.shift(1).rolling(window = 3, min_periods = 1).mean())
tournament_history['appearances_5yr'] = tournament_history.groupby('TeamID')['Round'].transform(lambda x: x.shift(1).rolling(window = 5, min_periods = 1).apply(lambda x: (x > 0).sum()))   
tournament_history 

In [ ]:
# Verifying the new features for a specific team (Duke my school & the best school OAT :))
tournament_history[tournament_history['TeamID'] == 1181]

In [ ]:
print(tournament_history.shape)
tournament_history

##### Winner Each Year Feature

In [ ]:
tourney_results.head()

In [ ]:
champions = tourney_results[['Season', 'WTeamID', 'DayNum']][tourney_results['DayNum'] == 154]
champions.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
champions['won_championship_last_year'] = 1
champions['Season'] += 1
champions.head()
champions.shape

In [ ]:
champions.drop(columns=['DayNum'], inplace=True)
tournament_history = pd.merge(tournament_history, champions, on=['Season', 'TeamID'], how='left')
tournament_history = tournament_history.fillna({'won_championship_last_year': 0})
tournament_history
tournament_history.shape

In [ ]:
# Verifying the new feature for a specific team (again Duke :) )
tournament_history[tournament_history['Season'].isin([1991,1992,1993]) & (tournament_history['TeamID'] == 1181)]

In [ ]:
print(tournament_history.shape)
tournament_history

## Step 5: Merging All Features

In [ ]:
#tournament_history = pd.merge(tournament_history, champions, on=['Season', 'TeamID'], how='left')
#tournament_history = tournament_history.fillna({'won_championship_last_year': 0})

team_features = pd.merge(tournament_history, season_averages, on=['Season', 'TeamID'], how='left')
team_features = pd.merge(team_features, seed_clean, on=['Season', 'TeamID'], how='left')
team_features = pd.merge(team_features, massey_pivot, on=['Season', 'TeamID'], how='left')

print(team_features.shape)
team_features.isnull().sum()    


In [ ]:
team_features

## Step 6: Creating table with proper features and target varible aligned

In [ ]:
tourney_results = pd.read_csv("../data/raw/MNCAATourneyCompactResults.csv")
tourney_results.head()

In [ ]:
matchups = tourney_results.copy()
np.random.seed(1) # Setting a seed for reproducibility
matchups['random_team'] = np.random.choice([True, False], size = len(matchups))
matchups.head()

In [ ]:
matchups['TeamA_Id'] = np.where(matchups['random_team'], matchups['WTeamID'], matchups['LTeamID'])
matchups['TeamB_Id'] = np.where(matchups['random_team'], matchups['LTeamID'], matchups['WTeamID'])
matchups['Results'] = np.where(matchups['random_team'], 1, 0)
# Cleaning up the matchups data
matchups_cleaned = matchups[['Season', 'DayNum', 'TeamA_Id', 'TeamB_Id', 'Results']]
matchups.head()

In [ ]:
team_features.head()

In [ ]:
final_features = pd.merge(matchups_cleaned, team_features, left_on = ['Season', 'TeamA_Id'], right_on = ['Season', 'TeamID'], suffixes = ('', '_A'), how='left')
final_features = pd.merge(final_features, team_features, left_on = ['Season', 'TeamB_Id'], right_on = ['Season', 'TeamID'], suffixes = ('', '_B'), how='left')
final_features.head()

In [ ]:
show_columns(final_features)

In [ ]:
# Cleaning up final_features
final_features = final_features.drop(columns=['TeamID', 'TeamID_B', 'DayNum_A', 'DayNum_B', 'Round', 'Round_B'], errors='ignore')
final_features.head()

In [ ]:
show_columns(final_features)

In [ ]:
final_features['Season'].value_counts().sort_index()

### Final Features

In [ ]:
final_features

## Step 7: rain/Validation/Test Split

In [1745]:
print('Results' in final_features.columns)
print(final_features['Season'].min(), final_features['Season'].max())

True
2003 2024


In [1747]:
# Getting proper date range
final_features = final_features[(final_features['Season'] >= 2003) & (final_features['Season'] <= 2024)].reset_index(drop=True)
final_features

,Season,DayNum,TeamA_Id,TeamB_Id,Results,round_last_year,round_avg_3yr,appearances_5yr,won_championship_last_year,Score,...,TO_B,Stl_B,Blk_B,PF_B,Win_B,Seed_B,DUN_B,MOR_B,POM_B,SAG_B
0,2003,134,1411,1421,0,0.0,0.000000,0.0,0.0,72.800000,...,16.206897,7.068966,3.000000,19.103448,0.448276,16.0,241.0,277.0,273.0,251.0
1,2003,136,1436,1112,0,0.0,0.000000,0.0,0.0,67.793103,...,14.785714,8.464286,4.214286,17.750000,0.892857,1.0,9.0,4.0,3.0,2.0
2,2003,136,1113,1272,1,0.0,0.000000,0.0,0.0,75.965517,...,13.793103,7.379310,5.068966,18.758621,0.793103,7.0,19.0,23.0,20.0,24.0
3,2003,136,1141,1166,1,0.0,0.000000,0.0,0.0,79.344828,...,13.363636,8.393939,4.454545,17.272727,0.878788,6.0,27.0,33.0,27.0,18.0
4,2003,136,1143,1301,1,2.0,1.000000,2.0,0.0,74.482759,...,14.200000,7.766667,3.066667,18.666667,0.600000,9.0,23.0,31.0,48.0,46.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,2024,146,1301,1181,1,1.0,0.333333,2.0,0.0,76.361111,...,9.375000,6.437500,3.656250,15.781250,0.750000,4.0,10.0,11.0,8.0,182.0
1378,2024,146,1397,1345,0,3.0,2.000000,5.0,0.0,79.468750,...,10.969697,5.666667,3.787879,14.363636,0.878788,1.0,3.0,6.0,3.0,182.0
1379,2024,152,1163,1104,1,6.0,3.000000,3.0,1.0,81.470588,...,11.812500,7.250000,4.062500,19.875000,0.656250,4.0,14.0,14.0,13.0,182.0
1380,2024,152,1301,1345,0,1.0,0.333333,2.0,0.0,76.361111,...,10.969697,5.666667,3.787879,14.363636,0.878788,1.0,3.0,6.0,3.0,182.0
